# Ablation — No Reasoning (RAG + direct label)

Removes the structured reasoning chain from the full pipeline.
The model receives the full profile-aware RAG exemplars (30-facet slices),
but outputs **only `high` or `low`** — no XML chain-of-thought.

Compare against full system (`reasoned_rag_def_oneshot_30f`) to isolate the contribution of structured reasoning.

**Requires:** `data/vector_db/essays_profile/` and `data/profile_db/essays_test/` from the main pipeline.

In [1]:
from pathlib import Path
import sys, os, json, time
from typing import Dict

import numpy as np
import pandas as pd

project_root = Path.cwd().resolve()
if not (project_root / "ptd_model").exists():
    project_root = (project_root / ".." / "..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from rag.profiler.store import ProfileStore
from rag.profiler.runner import build_profiles
from rag.profiler.prompts import FACETS, slice_profile_full_30
from rag.embedder import get_embedding
import rag.retriever as _retriever_mod
from rag.retriever import FeatureRAGRetriever

from ptd_model.prompts_other import SYS_PROMPT, DEF_ONESHOT_PROMPT
from ptd_model.prompts import TRAITS
from ptd_model.evaluate import evaluate
from utils.gpt_client import gpt_call
from utils.log import log_to_file
from utils.parser import extract_direct

print("Project root:", project_root)

Project root: F:\std\GR\code\model_x_ocean


## Configuration

In [2]:
test_csv        = str(project_root / "data/split/essays/test.csv")
test_profile_db = str(project_root / "data/profile_db/essays_test")
vector_db_dir   = str(project_root / "data/vector_db/essays_profile")
res_dir         = str(project_root / "result")
log_dir         = str(project_root / "log")

profiler_model = "gpt-4o-mini"
model_name     = "gpt-4o-mini"
max_new_tokens = 64
temperature    = 0.0
prompt_mode    = "ablation_no_reasoning_rag_30f"   # distinct save folder
top_k          = 5

TRAIT_NAMES = ("Openness to Experience", "Conscientiousness", "Extraversion", "Agreeableness", "Neuroticism")
TRAIT_CODES = {
    "Openness to Experience": "cOPN",
    "Conscientiousness":      "cCON",
    "Extraversion":           "cEXT",
    "Agreeableness":          "cAGR",
    "Neuroticism":            "cNEU",
}
TRAIT_SHORT_TO_FULL = {
    "Openness":          "Openness to Experience",
    "Conscientiousness": "Conscientiousness",
    "Extraversion":      "Extraversion",
    "Agreeableness":     "Agreeableness",
    "Neuroticism":       "Neuroticism",
}
TRAIT_TO_COLUMN = {
    "Openness":          "pred_cOPN",
    "Conscientiousness": "pred_cCON",
    "Extraversion":      "pred_cEXT",
    "Agreeableness":     "pred_cAGR",
    "Neuroticism":       "pred_cNEU",
}

test_df = pd.read_csv(test_csv)
print(f"Test: {len(test_df)} rows | mode={prompt_mode} | top_k={top_k}")

for fname in ("vectors.faiss", "vectors_meta.jsonl"):
    p = Path(vector_db_dir) / fname
    if not p.exists():
        raise RuntimeError(f"Missing {p}. Run rag_profile_half1_embed.ipynb first.")
print(f"Vector DB found at: {vector_db_dir}")

Test: 247 rows | mode=ablation_no_reasoning_rag_30f | top_k=5
Vector DB found at: F:\std\GR\code\model_x_ocean\data\vector_db\essays_profile


## Step 1 — Load test profiles (label-blind)

Re-uses the profiles already built by the main pipeline — no re-profiling needed.

In [3]:
test_store_path = Path(test_profile_db) / "profile_store.jsonl"
test_store = ProfileStore(str(test_store_path))
test_store.load()
needed = len(test_df) - sum(
    1 for i in range(len(test_df)) if test_store.has(f"user_{i}") and test_store.get(f"user_{i}").get("valid")
)
print(f"Test profiles in store: {len(test_store)}; missing: {needed}")

if needed > 0:
    test_store = build_profiles(
        data       = test_df,
        output_dir = test_profile_db,
        model_name = profiler_model,
        log_dir    = str(Path(log_dir) / "profiler_test"),
        use_labels = False,
    )
test_entries_by_idx = {
    int(e["user_id"].split("_")[1]): e for e in test_store.get_all() if e.get("valid")
}
print(f"Test profiles ready: {len(test_entries_by_idx)}")

Test profiles in store: 247; missing: 0
Test profiles ready: 247


## Step 2 — Profile-aware retriever (same as full system)

In [4]:
def render_full_profile_text(entry: Dict) -> str:
    raw = entry.get("raw") or ""
    if raw.strip():
        return raw
    facets = entry.get("facets", {})
    ling   = entry.get("linguistic", {})
    lines = ["[FACETS]"]
    for code, name, *_ in FACETS:
        f = facets.get(code, {})
        lines.append(f"{code} {name:<18}| {f.get('signal','')} | {f.get('evidence','')}")
    lines.append("\n[LINGUISTIC]")
    for k, v in ling.items():
        lines.append(f"{k}: {v}")
    return "\n".join(lines)


class ProfileRAGRetriever(FeatureRAGRetriever):
    def __init__(self, db_dir: str, test_profiles_by_idx: Dict, test_df: pd.DataFrame):
        super().__init__(db_dir=db_dir)
        self._use_finetuned = False
        self._test_profiles_by_idx = test_profiles_by_idx
        self._text_to_idx = {str(t): i for i, t in enumerate(test_df["text"].tolist())}

    def _embed_query_profile(self, query_text: str):
        idx = self._text_to_idx.get(str(query_text))
        if idx is None or idx not in self._test_profiles_by_idx:
            return self._embed_query(query_text)
        profile_text = render_full_profile_text(self._test_profiles_by_idx[idx])
        return np.array(self._embed_query(profile_text), dtype="float32")

    def build_similar_context_30f(self, posts: str, trait: str, top_k: int = 5) -> str:
        """Retrieve top_k exemplars and render full 30-facet profiles — same as full system."""
        query_emb  = self._embed_query_profile(posts)
        all_results = self._search(query_emb, top_k * 4)

        blocks, seen = [], 0
        for r in all_results:
            if trait not in r.get("trait_labels", {}):
                continue
            label    = r["trait_labels"][trait]
            features = r.get("features", {}) or {}
            profile  = features.get("profile") or {}
            slice_text = slice_profile_full_30(profile) if profile.get("facets") else "  (no profile available)"
            blocks.append(f"[Similar Profile {seen+1}] (label: {label})\n{slice_text}")
            seen += 1
            if seen >= top_k:
                break
        return "\n\n".join(blocks)


retriever = ProfileRAGRetriever(
    db_dir=vector_db_dir,
    test_profiles_by_idx=test_entries_by_idx,
    test_df=test_df,
)
print("[adapter] ProfileRAGRetriever ready.")

[retriever] mode='legacy'  dir='F:\\std\\GR\\code\\model_x_ocean\\data\\vector_db\\essays_profile'  hybrid=False
[adapter] ProfileRAGRetriever ready.


## Step 3 — Predict (RAG exemplars, direct label — no reasoning chain)

In [5]:
run_id       = time.strftime("%Y%m%d-%H%M%S")
safe_model   = model_name.replace(":", "_")
log_filepath = os.path.join(log_dir, safe_model, prompt_mode, f"{run_id}_log.txt")
output_dir   = os.path.join(res_dir, safe_model, prompt_mode, run_id)
os.makedirs(output_dir, exist_ok=True)

df = test_df.copy()
for col in TRAIT_TO_COLUMN.values():
    df[col] = None

n  = len(df)
t0 = time.time()
print(f"[predict] {n} records | mode={prompt_mode} | model={model_name}")

for idx, row in df.iterrows():
    text = row["text"]

    for trait_short, pred_col in TRAIT_TO_COLUMN.items():
        trait_full = TRAIT_SHORT_TO_FULL[trait_short]
        trait_defs = TRAITS.get(trait_short, {})

        # --- ablation: full RAG context, but direct-label prompt (no CoT) ---
        similar_context = retriever.build_similar_context_30f(
            posts=text, trait=trait_full, top_k=top_k
        )
        usr_prompt = DEF_ONESHOT_PROMPT.format(
            trait_name=trait_short,
            definition_high=trait_defs.get("high", ""),
            definition_low=trait_defs.get("low", ""),
            top_k=top_k,
            similar_context=similar_context,
        )

        formatted = usr_prompt.replace("<text>", text)
        output = gpt_call(formatted, SYS_PROMPT, model_name, max_new_tokens, temperature)
        log_to_file(log_filepath, SYS_PROMPT, formatted, output, f"{idx}-{trait_short}")

        df.at[idx, pred_col] = extract_direct(output.strip())

    if (idx + 1) % 10 == 0:
        print(f"  [predict] {idx + 1}/{n} done.")

elapsed = time.time() - t0
prediction_csv = os.path.join(output_dir, "predictions.csv")
df.to_csv(prediction_csv, index=False)
print(f"[predict] Finished in {elapsed:.1f}s -> {prediction_csv}")

[predict] 247 records | mode=ablation_no_reasoning_rag_30f | model=gpt-4o-mini
[embedder] Loading embedding model: nomic-ai/nomic-embed-text-v1.5


<All keys matched successfully>


[retriever] Legacy index loaded (1974 vectors).
  [predict] 10/247 done.
  [predict] 20/247 done.
  [predict] 30/247 done.
  [predict] 40/247 done.
  [predict] 50/247 done.
  [predict] 60/247 done.
  [predict] 70/247 done.
  [predict] 80/247 done.
  [predict] 90/247 done.
  [predict] 100/247 done.
  [predict] 110/247 done.
  [predict] 120/247 done.
  [predict] 130/247 done.
  [predict] 140/247 done.
  [predict] 150/247 done.
  [predict] 160/247 done.
  [predict] 170/247 done.
  [predict] 180/247 done.
  [predict] 190/247 done.
  [predict] 200/247 done.
  [predict] 210/247 done.
  [predict] 220/247 done.
  [predict] 230/247 done.
  [predict] 240/247 done.
[predict] Finished in 3271.7s -> F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\ablation_no_reasoning_rag_30f\20260531-013554\predictions.csv


## Step 4 — Evaluate

In [ ]:
evaluation = evaluate(
    prediction_csv = prediction_csv,
    model_name     = model_name,
    res_dir        = res_dir,
    run_time       = elapsed,
    prompt_mode    = prompt_mode,
    run_id         = run_id,
)

print("Summary CSV:", evaluation["summary_csv"])
print(f"Failed predictions: {evaluation['fail_count']} / {evaluation['n_records']}")
summary_df = pd.read_csv(evaluation["summary_csv"])
display(summary_df[["trait", "n_samples", "accuracy", "macro_f1", "weighted_f1"]]
        .sort_values("accuracy", ascending=False)
        .reset_index(drop=True))

Loaded predictions from F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\ablation_no_reasoning_rag_30f\20260531-013554\predictions.csv
Saved evaluation summary to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\ablation_no_reasoning_rag_30f\20260531-013554\evaluation_summary.csv
Saved Openness report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\ablation_no_reasoning_rag_30f\20260531-013554\Openness_classification_report.txt
Saved Conscientiousness report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\ablation_no_reasoning_rag_30f\20260531-013554\Conscientiousness_classification_report.txt
Saved Extraversion report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\ablation_no_reasoning_rag_30f\20260531-013554\Extraversion_classification_report.txt
Saved Agreeableness report to F:\std\GR\code\model_x_ocean\result\gpt-4o-mini\ablation_no_reasoning_rag_30f\20260531-013554\Agreeableness_classification_report.txt
Saved Neuroticism report to F:\std\GR\code\model_x_ocean\result\gpt-4

,trait,n_samples,accuracy,macro_f1,weighted_f1
0,Openness,247,0.599190,0.597050,0.596218
1,Neuroticism,247,0.591093,0.553121,0.553649
2,Extraversion,247,0.562753,0.553794,0.552002
3,Agreeableness,247,0.550607,0.537227,0.531174
4,Conscientiousness,247,0.526316,0.398101,0.394726


: 